In [1]:
%pip install -r requirements.txt

  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached cfedemands-0.8.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached ligonlibrary-0.2.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached consumerdemands-0.5.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached gspread_pandas-3.3.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached python_magic-0.4.27-py2.py3-none-any.whl.metadata (5.8 kB)
  Using cached thefuzz-0.22.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached rapidfuzz-3.14.5-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.3 MB)
Using cached cfedemands-0.8.2-py3-none-any.whl (98 kB)
Using cached ligonlibrary-0.2.0-py3-none-any.whl (19 kB)
Using cached consumerdemands-0.5.0-py3-none-any.whl (14 kB)
Using cached gspread_pandas-3.3.

In [2]:
import pandas as pd
from ligonlibrary.sheets import read_sheets
import unicodedata as unicodedata
import re
import difflib

# import library for panama csv 
#panama food expenditures with household distributions ; the main df
panama_household_dist_df = "1QKSfYFQu8xR4I-vHfPK00mO-1XjqrQgVpCGGpDrustI" 
x = read_sheets(panama_household_dist_df,sheet='Food Expenditures (2003)')

x.head()

,i,t,m,j,Expenditure
0,20031000,2003,Herrera,Aceite Vegetal,2.8
1,20031000,2003,Herrera,Ahí Verde,1.2
2,20031000,2003,Herrera,Ajo,1.0
3,20031000,2003,Herrera,Apio,0.3
4,20031000,2003,Herrera,Arroz,10.5


In [3]:
panama_foods_nutrients_df = "1C3XN4V08r4JAfLstg94EgOrrlBqw4E48w2SYjK8qxnY" 
y = read_sheets(panama_foods_nutrients_df,sheet='central_american_fct - FCT', nheaders=3)
y.head()

,Nutrient,Agua\n%,Energía,Proteina,Grasa Total,Carbo-\nhidratos,Fibra Diet. \ntotal,Ceniza,Calcio,Fosforo,...,Colesterol,Potasio,Sodio,Zinc,Magnesio,Vit. B6,Vit. B12,Ac. Fólico,Folato \nEquiv. FD,Fracción \nComestible
,Unit,,Kcal.,g,g,g,g,g,mg,mg,...,mg,mg,mg,mg,mg,mg,mcg,mcg,mcg,%
,Name,,,,,,,,,,...,,,,,,,,,,
Code,,,,,,,,,,,,,,,,,,,,,
,01. PRODUCTOS LACTEOS Y SIMILARES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1001,"CREMA, ESPESA",57.71,345.0,2.05,37.00,2.79,0.0,0.45,65.0,62.0,...,137.0,75.0,38.0,0.23,7.0,0.03,0.18,0.0,4.0,1.0
1002,"CREMA, RALA",73.75,195.0,2.70,19.31,3.66,0.0,0.58,96.0,80.0,...,66.0,122.0,40.0,0.27,9.0,0.03,0.22,0.0,2.0,1.0
1003,"CREMA, SUSTITUTO NO LÁCTEO, EN POLVO",2.21,546.0,4.79,35.48,54.88,0.0,2.64,22.0,422.0,...,0.0,812.0,181.0,0.51,4.0,0.00,0.00,0.0,0.0,1.0
1004,"LECHE DE BURRA, FLUIDA",90.20,43.0,1.70,1.20,6.50,NaN,0.40,126.0,57.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [4]:
?read_sheets

Signature:
read_sheets(
    key: 'str | Mapping[str, str]',
    json_creds: 'Optional[str]' = None,
    sheet: 'Optional[str | int]' = None,
    force_numeric: 'bool' = True,
    nheaders: 'int' = 1,
)
Docstring:
Read a Google Sheet via key or URL, optionally using service-account credentials.

Parameters mirror the historical interface used in course material.
File:      /srv/conda/lib/python3.11/site-packages/ligonlibrary/sheets.py
Type:      function

In [4]:
grouped_x = x.groupby('i')['j'].apply(list).reset_index()
grouped_x.head()

,i,j
0,2003199,"[Aceite Vegetal, Ahí Verde, Apio, Arroz, Azúca..."
1,2003201,"[Arroz, Azúcar, Café Y Té, Carne De Res, Galle..."
2,2003202,"[Arroz, Azúcar, Café Y Té, Cebolla, Galletas /..."
3,2003203,"[Aceite Vegetal, Ajo, Arroz, Azúcar, Café Y Té..."
4,2003204,"[Aceite Vegetal, Ajo, Arroz, Azúcar, Café Y Té..."


In [5]:
# function for cleaning ONE food name
def clean_one_food(s):
    if pd.isna(s):
        return None
    s = str(s).upper().strip()
    # remove accents
    s = ''.join(
        c for c in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(c)
    )
# replacing separators with spaces
    s = s.replace('/', ' ')
    s = s.replace('-', ' ')
    s = s.replace(',', ' ')
    s = s.replace('.', ' ')
# removing parentheses/brackets
    s = re.sub(r'[\(\)\[\]\{\}]', ' ', s)
# keeping only letters, numbers, spaces
    s = re.sub(r'[^A-Z0-9\s]', ' ', s)
# normalizing common filler words
    replacements = {
        'DE LA ': ' ',
        'DE LAS ': ' ',
        'DE LOS ': ' ',
        'DE EL ': ' ',
        ' DEL ': ' ',
        ' DE ': ' ',
        ' Y ': ' '
    }
    s = f' {s} '
    for old, new in replacements.items():
        s = s.replace(old, f' {new} ')
    s = s.strip()
# collapsing multiple spaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s


# function for cleaning the LIST of foods
def clean_food_list(food_list):
    if not isinstance(food_list, list):
        return None
    return [clean_one_food(food) for food in food_list if pd.notna(food)]


# apply to grouped table
grouped_x['j_clean'] = grouped_x['j'].apply(clean_food_list)

grouped_x.head()

,i,j,j_clean
0,2003199,"[Aceite Vegetal, Ahí Verde, Apio, Arroz, Azúca...","[ACEITE VEGETAL, AHI VERDE, APIO, ARROZ, AZUCA..."
1,2003201,"[Arroz, Azúcar, Café Y Té, Carne De Res, Galle...","[ARROZ, AZUCAR, CAFE TE, CARNE RES, GALLETAS P..."
2,2003202,"[Arroz, Azúcar, Café Y Té, Cebolla, Galletas /...","[ARROZ, AZUCAR, CAFE TE, CEBOLLA, GALLETAS PAS..."
3,2003203,"[Aceite Vegetal, Ajo, Arroz, Azúcar, Café Y Té...","[ACEITE VEGETAL, AJO, ARROZ, AZUCAR, CAFE TE, ..."
4,2003204,"[Aceite Vegetal, Ajo, Arroz, Azúcar, Café Y Té...","[ACEITE VEGETAL, AJO, ARROZ, AZUCAR, CAFE TE, ..."


In [6]:
y.iloc[:, 0]

Code
            01.  PRODUCTOS LACTEOS Y SIMILARES
1001                             CREMA, ESPESA
1002                               CREMA, RALA
1003     CREMA, SUSTITUTO NO LÁCTEO,  EN POLVO
1004                    LECHE DE BURRA, FLUIDA
                         ...                  
22021                              SAL DE MESA
22022         SAZONADOR MOLIDO NO ESPECIFICADO
22023                    SAZONADOR, PARA POLLO
22024                            TOMILLO, SECO
22025                       VAINILLA, EXTRACTO
Name: (Nutrient, Unit, Name), Length: 1466, dtype: str

In [ ]:
foods_long = grouped_x[['i', 'j_clean']].explode('j_clean').rename(columns={'i': 'household_id', 'j_clean': 'searched_food'})
y_names = y.iloc[:, 0].astype(str)

results = []

for food in foods_long['searched_food'].dropna().unique():
    mask = y_names.str.contains(food, regex=False, na=False)
    matched_y = y[mask].copy()
    
    if not matched_y.empty:
        temp_foods = foods_long[foods_long['searched_food'] == food]
        for _, food_row in temp_foods.iterrows():
            temp = matched_y.copy()
            temp['household_id'] = food_row['household_id']
            temp['searched_food'] = food
            results.append(temp)

final_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
final_df.head()